# Sidekick worker and evaluator routing: practice

Worker may request tools → otherwise evaluator checks success criteria → stop for success/clarification or return feedback to the worker → retain state by thread ID.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [4_langgraph/4_lab4.ipynb](../../../../../4_langgraph/4_lab4.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Structured output validates shape

Pydantic models name the fields you expect and check constraints. This helps callers use results reliably, but a valid field can still contain a false claim.

**Predict:** Will confidence=2 pass the constraint?

In [ ]:
from pydantic import BaseModel, Field, ValidationError
class Report(BaseModel):
    summary: str
    confidence: float = Field(ge=0, le=1)
report = Report(summary="Fictional result", confidence=0.8)
print(report.model_dump())
try:
    Report(summary="Bad shape", confidence=2)
except ValidationError:
    print("Invalid confidence rejected")
else:
    raise AssertionError("Constraint was not enforced")

**Check:** No. The value must be between zero and one. Validation checks the constraint, not the truth of the summary.

**Change one thing:** Add a field named follow_up_questions with type list[str]. Construct a valid object using it.

## 2. Conditional routing makes the next step explicit

A route is a function of state. A worker may need tools, evaluation, another attempt, or user clarification. This exercise checks all branch labels without calling any model.

**Predict:** Which route wins when a tool call is present?

In [ ]:
def worker_route(state):
    return "tools" if state["tool_calls"] else "evaluator"
def evaluation_route(state):
    if state["success"] or state["needs_user"]:
        return "END"
    return "worker"
assert worker_route({"tool_calls": ["lookup"]}) == "tools"
assert worker_route({"tool_calls": []}) == "evaluator"
for state in [{"success": True, "needs_user": False},
              {"success": False, "needs_user": True},
              {"success": False, "needs_user": False}]:
    print(evaluation_route(state))

**Check:** Tools first. Evaluation can stop for success or required user input, and otherwise return to the worker.

**Change one thing:** Add an attempts field and route to END after a fixed limit. Explain why asking for clarification is a valid stop condition.

## 3. Evaluate, give feedback, and bound revision

A worker produces a draft and an evaluator judges it against criteria. Feedback becomes input for another attempt. The evaluator here is a small deterministic teaching rule, not a general quality checker.

**Predict:** Is the first draft acceptable? How many attempts will be allowed?

In [ ]:
def evaluate(text):
    return {"accepted": "example:" in text.lower(), "feedback": "Include an example."}
attempts = []
for attempt in range(2):
    draft = "A list stores ordered values." if attempt == 0 else "A list stores ordered values. Example: [1, 2]"
    decision = evaluate(draft)
    attempts.append(decision["accepted"])
    if decision["accepted"]:
        break
print(attempts, draft)
assert attempts == [False, True]

**Check:** The first is rejected and the second accepted. Unlike the Lab 3 one-retry source, this teaching example evaluates the revision too.

**Change one thing:** Make the second draft fail too. Add an explicit message when the finite retry limit is reached.

## 4. Build a real local LangGraph without a model

A graph is state plus nodes plus edges. Nodes return state updates. Compilation builds the executable workflow; invoke runs it. This exercises the installed LangGraph package locally.

**Predict:** Which value will the uppercase node return?

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class State(TypedDict):
    text: str
def uppercase(state):
    return {"text": state["text"].upper()}
builder = StateGraph(State)
builder.add_node("uppercase", uppercase)
builder.add_edge(START, "uppercase")
builder.add_edge("uppercase", END)
graph = builder.compile()
result = graph.invoke({"text": "hello"})
print(result)
assert result["text"] == "HELLO"

**Check:** HELLO. Graphs can orchestrate ordinary Python functions; a language model is optional.

**Change one thing:** Add a second node that appends an exclamation mark. Wire it between uppercase and END.

## 5. Reducer vs. checkpoints vs. conversation identity

A reducer combines state updates. A checkpointer retains state across invocations. A thread_id selects a conversation. This uses SQLite in memory, so it creates no database file.

**Predict:** Does a different thread_id share the same list?

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
class MemoryState(TypedDict):
    turns: Annotated[list[str], add]
builder = StateGraph(MemoryState)
builder.add_node("reply", lambda state: {"turns": ["assistant reply"]})
builder.add_edge(START, "reply")
builder.add_edge("reply", END)
with SqliteSaver.from_conn_string(":memory:") as saver:
    graph = builder.compile(checkpointer=saver)
    config = {"configurable": {"thread_id": "practice-1"}}
    graph.invoke({"turns": ["first user turn"]}, config)
    result = graph.invoke({"turns": ["second user turn"]}, config)
    separate = graph.invoke({"turns": ["new user"]}, {"configurable": {"thread_id": "practice-2"}})
    print(result["turns"], separate["turns"])
    assert len(result["turns"]) == 4 and len(separate["turns"]) == 2

**Check:** No. Same ID continues a thread; a different ID selects another. In-memory SQLite here disappears when the connection closes.

**Change one thing:** Replace :memory: with a deliberately chosen practice database only if you want persistence. Explain when it would survive process exit.

## 6. A callback receives a message and history

A UI invokes your function and displays its return value. Earlier turns are an input, not automatic model memory. This practice calls the callback directly without starting a server.

**Predict:** What changes when history contains two old messages?

In [ ]:
def chat(message, history):
    return f"You asked: {message}. Earlier messages: {len(history)}."
first = chat("Hello", [])
second = chat("Explain more", [{"role":"user", "content":"Hello"}, {"role":"assistant", "content":first}])
print(first, second, sep="\n")
assert "Earlier messages: 2" in second

**Check:** The count becomes two. Gradio wiring is gr.ChatInterface(chat, type="messages"); launch starts the server.

**Change one thing:** Make the callback return an uppercase version of the question. Compare returning a value with printing it.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.